# arxiv2md pipeline

用来实验 arxiv 论文 html 转化为 md 的 pipeline

思路：不碰 PDF，直接抓 arXiv 官方 HTML（LaTeXML 从 TeX 源码生成，`arxiv.org/html/<id>`，
没有时回退 ar5iv），再按 `ltx_*` class 的结构序列化成 Markdown。库入口
`ingest_paper` 一步做完；这里把它**拆开逐步跑**，每一步看一眼中间产物：

    arxiv id ──parse_arxiv_input──▶ ArxivQuery (html_url / ar5iv_url)
             ──fetch_arxiv_html───▶ 原始 HTML（本地缓存）
             ──parse_arxiv_html───▶ title / authors / abstract / SectionNode 树（每节带 html 片段）
             ──convert_fragment_to_markdown（逐节）──▶ SectionNode.markdown
             ──format_paper + unlink_dangling_fragment_links──▶ 整篇 md
             ──localize_images────▶ 图片落盘、链接改相对路径

## step1 预设 arxiv id

In [ ]:
# 2603.13774v1 AgenticScholar 作为样例

arxiv_id = "2603.13774v1" 

import os
from pathlib import Path

# ── 运行参数 ──
PROXY = None                  # 例 "http://127.0.0.1:7890"；None 则沿用环境里已有的 HTTP(S)_PROXY
REFETCH_HTML = False          # True：忽略已落盘的 source.html，重新抓
REDOWNLOAD_IMAGES = False     # True：清掉已下载的图片全部重下
IMAGE_CONCURRENCY = 8         # 图片并发下载数
CHECK_ONE_SHOT = True         # 3.6 对照一步版 ingest_paper；arxiv2md 的 HTML 缓存过期时会再请求一次

if PROXY:
    for k in ("HTTP_PROXY", "HTTPS_PROXY", "http_proxy", "https_proxy"):
        os.environ[k] = PROXY
# arxiv2md 在 import 时读这些配置（默认超时 10s、重试 2 次，走代理时偏紧），必须在 step2 之前设
os.environ.setdefault("ARXIV2MD_FETCH_TIMEOUT_S", "30")
os.environ.setdefault("ARXIV2MD_FETCH_MAX_RETRIES", "3")

OUT = Path.cwd() / "_scratch/arxiv2md" / arxiv_id.replace("/", "_")   # 本篇产物目录，也是断点续跑的依据

### 产物检查与网络

先看 `OUT` 里已有什么，据此判断后面哪些步骤要联网：

- `source.html` + `source_url.txt` 已在 → 3.2 直接复用，不再抓（不受 arxiv2md 缓存 24h TTL 的影响）；
- `images/` 里已完成的图片 → step4 跳过；残留的 `*.part` 是上次中断的半截文件，会用 HTTP Range 续传；
- `<id>.md` / `meta.json` 由本地转换生成，只要几秒，每次都重新写。

代理：httpx 在建 client 时读 `HTTP(S)_PROXY` / `ALL_PROXY` / `NO_PROXY` 环境变量，arxiv2md 和下面的
图片下载都走这条路，所以设一次环境变量就全部生效。SOCKS 代理需要 `httpx[socks]`，当前环境没装，用 HTTP 代理。
有东西要下载时，顺手探测一下 arxiv.org 是否可达、延迟多少。

In [ ]:
import re, time
import httpx

REMOTE_IMG_RE = re.compile(r"!\[(?:\\.|[^\\\]])*\]\(https?://")   # 仍指向远程的图片链接（只数图片，不数引文等普通链接）

def product_status(out: Path) -> dict:
    html_ok = (out / "source.html").is_file() and (out / "source_url.txt").is_file()
    img_dir = out / "images"
    done = [p for p in img_dir.glob("*") if not p.name.endswith(".part")] if img_dir.is_dir() else []
    parts = list(img_dir.glob("*.part")) if img_dir.is_dir() else []
    md_path = out / f"{out.name}.md"
    remote = len(REMOTE_IMG_RE.findall(md_path.read_text(encoding="utf-8"))) if md_path.is_file() else None
    return {"html": html_ok, "images_done": len(done), "images_part": len(parts),
            "md": md_path.is_file(), "md_remote_images": remote, "meta": (out / "meta.json").is_file()}

status = product_status(OUT)
print(OUT)
print(f"  source.html : {'✓ 复用' if status['html'] and not REFETCH_HTML else '✗ 需要抓取'}")
print(f"  images/     : 已完成 {status['images_done']} 张，半截 .part {status['images_part']} 个"
      + ("（将全部重下）" if REDOWNLOAD_IMAGES else ""))
print(f"  md / meta   : {'✓' if status['md'] else '✗'} / {'✓' if status['meta'] else '✗'}"
      + (f"（上次 md 里仍有 {status['md_remote_images']} 张图是远程链接）" if status["md_remote_images"] else ""))

proxy = next((os.environ[k] for k in ("HTTPS_PROXY", "https_proxy", "ALL_PROXY", "all_proxy") if os.environ.get(k)), None)
print("代理:", proxy or "无（直连）")

async def probe(url="https://arxiv.org/", timeout=10.0):
    t0 = time.monotonic()
    try:
        async with httpx.AsyncClient(timeout=timeout, follow_redirects=True) as c:
            r = await c.head(url)
        return f"HTTP {r.status_code}, {time.monotonic() - t0:.2f}s"
    except httpx.HTTPError as exc:
        return f"不可达 {type(exc).__name__}: {exc}"

need_network = REFETCH_HTML or not status["html"] or REDOWNLOAD_IMAGES or status["md_remote_images"] != 0
print("arxiv.org 探测:", await probe() if need_network else "跳过（产物齐全，本次无需联网）")

## step2 导入 arxiv2md packages

`arxiv2md` 是仓库里 vendored 的 fork（`packages/arxiv2md`，workspace 成员，共享 `.venv` 里已装好）。
顶层只暴露 `ingest_paper` / `ingest_paper_sync`；为了拆步演示，这里直接导入各阶段的内部模块。

In [ ]:
import logging
from pathlib import Path

import arxiv2md
from arxiv2md.query_parser import parse_arxiv_input            # id/URL → ArxivQuery
from arxiv2md.fetch import fetch_arxiv_html                    # 抓 HTML（带缓存、ar5iv 回退）
from arxiv2md.html_parser import parse_arxiv_html              # HTML → 元数据 + 章节树
from arxiv2md.markdown import (
    convert_fragment_to_markdown,                              # 单节 HTML 片段 → md
    unlink_dangling_fragment_links,                            # 悬空的 [x](#frag) 降级为纯文本
)
from arxiv2md.output_formatter import format_paper             # 拼装整篇 md + summary + 章节树
from arxiv2md.images import localize_images                    # 下载图片、改写链接
from arxiv2md.config import ARXIV2MD_CACHE_PATH

logging.getLogger("httpx").setLevel(logging.WARNING)       # 每张图一条 HTTP INFO，太吵

REPO = Path.cwd()
while not (REPO / "AGENTS.md").exists():                       # notebook 可能从任意目录启动
    REPO = REPO.parent

print("arxiv2md from:", Path(arxiv2md.__file__).parent.relative_to(REPO))
print("HTML cache   :", ARXIV2MD_CACHE_PATH)

## step3 处理

### 3.1 解析 id

`parse_arxiv_input` 接受裸 id、带版本号的 id、abs/pdf/html URL，统一成 `ArxivQuery`。
带版本号就锁定版本（`html/2603.13774v1`），不带则取最新版 —— 为了可复现，尽量带版本号。

In [ ]:
query = parse_arxiv_input(arxiv_id)
print("arxiv_id :", query.arxiv_id)
print("version  :", query.version)
print("html_url :", query.html_url)
print("ar5iv_url:", query.ar5iv_url)

### 3.2 抓 HTML

先试 `arxiv.org/html/<id>`，404 时回退 ar5iv；`source_url` 记录实际用的是哪个。

两层缓存：`OUT/source.html` 在就直接读（本篇的产物，不过期）；否则调 `fetch_arxiv_html`
（它自己还有一层 `ARXIV2MD_CACHE_PATH` 缓存，TTL 24h），抓到后**立刻落盘**，后面步骤挂了重跑也不用再抓。
Jupyter 里已有事件循环，所以直接 `await`。

In [ ]:
html_path, url_path = OUT / "source.html", OUT / "source_url.txt"

if html_path.is_file() and url_path.is_file() and not REFETCH_HTML:
    html = html_path.read_text(encoding="utf-8")
    source_url = url_path.read_text(encoding="utf-8").strip()
    print("复用已落盘的 HTML")
else:
    t0 = time.monotonic()
    html, source_url = await fetch_arxiv_html(
        query.html_url,
        arxiv_id=query.arxiv_id,
        version=query.version,
        ar5iv_url=query.ar5iv_url,
        use_cache=True,
    )
    OUT.mkdir(parents=True, exist_ok=True)
    html_path.write_text(html, encoding="utf-8")
    url_path.write_text(source_url, encoding="utf-8")
    print(f"抓取完成 {time.monotonic() - t0:.1f}s，已落盘")

print("source_url:", source_url)
print(f"HTML {len(html):,} chars | ltx_section {html.count('ltx_section')} 处 | "
      f"ltx_figure {html.count('ltx_figure')} 处 | ltx_equation {html.count('ltx_equation')} 处")

### 3.3 HTML → 元数据 + 章节树

`parse_arxiv_html` 找到 `article.ltx_document`，抽 title / authors / abstract，
再按 `h2`–`h6` 标题切出章节树。此时每个 `SectionNode` 只有 `html` 片段，`markdown` 还是空的。

In [ ]:
parsed = parse_arxiv_html(html)
print("title  :", parsed.title)
print("authors:", parsed.authors)
print("abstract:", (parsed.abstract or "")[:300], "...")

def show_tree(nodes, indent=0):
    for n in nodes:
        size = len(n.html or "")
        print(f"{'  ' * indent}- [{n.anchor}] {n.title}  (h{n.level}, html {size:,} chars)")
        show_tree(n.children, indent + 1)

show_tree(parsed.sections)

### 3.4 逐节 HTML → Markdown

核心转换在 `convert_fragment_to_markdown`：MathML → LaTeX（行间公式按 LaTeXML 表结构逐条出 `$$`）、
表格按 rowspan/colspan 展网格、figure 出 `![图注](url)`、引文保留为 `[n](#bib.bibN)` 并插锚点、
代码/算法出围栏代码块。`base_url` 用于把图片相对路径解析成绝对 URL。

这里挑一节单独看转换前后，再把所有节（含子节）都填上 `markdown`。

In [ ]:
sample = parsed.sections[1]                    # 换成想看的节
sample_md = convert_fragment_to_markdown(sample.html, base_url=source_url)
print(f"【{sample.title}】 html {len(sample.html):,} chars → md {len(sample_md):,} chars\n")
print(sample_md[:1500])

In [ ]:
def populate(node):
    if node.html:
        node.markdown = convert_fragment_to_markdown(node.html, base_url=source_url)
    for child in node.children:
        populate(child)

for sec in parsed.sections:                    # 这里不做章节过滤；需要时用 arxiv2md.sections.filter_sections
    populate(sec)

### 3.5 拼装整篇

`format_paper` 按树序拼出 `## Abstract` + 各节（标题上方插 `<a id=…>` 锚点），同时给出 summary 和章节树。
之后 `unlink_dangling_fragment_links` 把目标锚点不存在的页内链接降级为纯文本，保证没有死链。

In [ ]:
result = format_paper(
    arxiv_id=query.arxiv_id,
    version=query.version,
    title=parsed.title,
    authors=parsed.authors,
    abstract=parsed.abstract,
    sections=parsed.sections,
    include_toc=False,
    include_abstract_in_tree=parsed.abstract is not None,
    include_frontmatter=True,
)
content = unlink_dangling_fragment_links(result.content)

print(result.summary)
print()
print(result.sections_tree)

In [ ]:
import re

lines = content.splitlines()
print(f"md {len(content):,} chars | 标题 {sum(l.startswith('#') for l in lines)} 个 | "
      f"图片 {content.count('![')} 处 | $$ 块 {content.count('$$') // 2} 个 | "
      f"表格行 {sum(l.startswith('|') for l in lines)} 行 | "
      f"引文链接 {len(re.findall(r'\]\(#bib\.', content))} 处 | "
      f"代码块 {content.count('```') // 2} 个")
print()
print(content[:2000])

### 3.6 对照：一步版 `ingest_paper`

上面拆开的流程应当与库入口的输出逐字一致。不一致说明拆步时漏了参数。
一步版走的是 arxiv2md 自己的 HTML 缓存，过期了会重新请求一次，嫌慢就把 `CHECK_ONE_SHOT` 关掉。

In [ ]:
if CHECK_ONE_SHOT:
    one_shot = await arxiv2md.ingest_paper(arxiv_id, include_frontmatter=True)
    assert one_shot.content == content, "拆步结果与 ingest_paper 不一致"
    print("一致 ✓")
else:
    print("跳过")

## step4 落盘存储

输出到 `_scratch/arxiv2md/<arxiv_id>/`（探索产物，不进 `data/processed/`），布局与 `01_ocr_pdf` 对齐：

    <arxiv_id>.md     正文（YAML frontmatter + md），图片链接改为 images/ 下的相对路径
    images/           下载到本地的图片；下载失败的保留远程 URL
    source.html       抓到的原始 HTML（3.2 已写），留作回溯
    source_url.txt    HTML 实际来源（arxiv.org 或 ar5iv）
    meta.json         溯源：来源 URL、HTML sha256、arxiv2md 所在 git commit、时间、计数

### 4.1 下载图片（并发 + 断点续传）

库里的 `localize_images` 是串行下载、每次全量重下。这里换成自己的下载循环，文件名规则仍用库的
`_image_filename`，与 `--download-images` 的产物一致：

- **跳过**：目标文件已存在且非空，视为完成；
- **续传**：先写 `<name>.part`，完成后才改名。中断后留下的 `.part` 下次带 `Range: bytes=<已有字节>-` 续传；
  服务器回 206 就追加，回 200（不支持 Range）就从头覆盖，回 416 说明 `.part` 其实已经完整；
- **重试**：单张失败最多重试 3 次（指数退避），404 不重试；最终失败的那张在 md 里保留远程 URL，重跑本 cell 会再补。

In [ ]:
import asyncio, shutil
from arxiv2md.images import _MD_IMAGE_RE, _image_filename     # 与库的命名规则保持一致
from arxiv2md.config import ARXIV2MD_FETCH_TIMEOUT_S, ARXIV2MD_USER_AGENT

IMG_DIR = OUT / "images"
if REDOWNLOAD_IMAGES and IMG_DIR.exists():
    shutil.rmtree(IMG_DIR)
IMG_DIR.mkdir(parents=True, exist_ok=True)

urls = list(dict.fromkeys(m.group(2) for m in _MD_IMAGE_RE.finditer(content)))
used_names: set[str] = set()
targets = {u: IMG_DIR / _image_filename(u, used_names) for u in urls}


async def download(client: httpx.AsyncClient, url: str, dest: Path, retries: int = 3) -> str:
    if dest.is_file() and dest.stat().st_size > 0:
        return "skip"
    part = dest.with_name(dest.name + ".part")
    last_exc = None
    for attempt in range(retries + 1):
        offset = part.stat().st_size if part.exists() else 0
        headers = {"Range": f"bytes={offset}-"} if offset else {}
        try:
            async with client.stream("GET", url, headers=headers) as r:
                if r.status_code == 416 and offset:            # .part 已经是完整文件
                    part.rename(dest)
                    return "resumed"
                r.raise_for_status()
                resumed = r.status_code == 206
                with part.open("ab" if resumed else "wb") as f:
                    async for chunk in r.aiter_bytes():
                        f.write(chunk)
            part.rename(dest)
            return "resumed" if resumed else "ok"
        except (httpx.HTTPError, OSError) as exc:
            last_exc = exc
            if isinstance(exc, httpx.HTTPStatusError) and exc.response.status_code == 404:
                break
            await asyncio.sleep(0.5 * 2 ** attempt)
    return f"FAIL {type(last_exc).__name__}: {last_exc}"


sem = asyncio.Semaphore(IMAGE_CONCURRENCY)

async def bounded(client, url, dest):
    async with sem:
        return url, await download(client, url, dest)

t0 = time.monotonic()
async with httpx.AsyncClient(timeout=httpx.Timeout(ARXIV2MD_FETCH_TIMEOUT_S),
                             headers={"User-Agent": ARXIV2MD_USER_AGENT},
                             follow_redirects=True) as client:
    outcomes = dict(await asyncio.gather(*(bounded(client, u, d) for u, d in targets.items())))

tally = {k: sum(v.startswith(k) for v in outcomes.values()) for k in ("ok", "resumed", "skip", "FAIL")}
print(f"{len(urls)} 张图 | 新下 {tally['ok']} | 续传 {tally['resumed']} | 跳过 {tally['skip']} | "
      f"失败 {tally['FAIL']} | {time.monotonic() - t0:.1f}s")
for u, v in outcomes.items():
    if v.startswith("FAIL"):
        print("  ", u, "→", v)

# 只改写已落地的图片链接；失败的保留远程 URL
local = {u: f"{IMG_DIR.name}/{d.name}" for u, d in targets.items() if d.is_file()}
md_local = _MD_IMAGE_RE.sub(lambda m: f"![{m.group(1)}]({local.get(m.group(2), m.group(2))})", content)

### 4.2 写 md 与 meta

纯本地操作，每次覆盖。`meta.json` 里的 `images_local < images_total` 说明还有图没下到，重跑 4.1 即可补齐。

In [ ]:
import hashlib, json, subprocess
from datetime import datetime, timezone

md_path = OUT / f"{OUT.name}.md"
md_path.write_text((result.frontmatter or "") + md_local + "\n", encoding="utf-8")

git_sha = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO,
                         capture_output=True, text=True).stdout.strip()
meta = {
    "arxiv_id": query.arxiv_id,
    "version": query.version,
    "source_url": source_url,
    "html_sha256": hashlib.sha256(html.encode("utf-8")).hexdigest(),
    "title": parsed.title,
    "authors": parsed.authors,
    "converter": {"name": "arxiv2md (vendored, packages/arxiv2md)", "git_sha": git_sha},
    "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "counts": {
        "md_chars": len(md_local),
        "images_total": len(urls),
        "images_local": len(local),
        "display_math": md_local.count("$$") // 2,
    },
    "outputs": {"md": md_path.name, "md_sha256": hashlib.sha256(md_path.read_bytes()).hexdigest()},
}
(OUT / "meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=1), encoding="utf-8")

print(OUT.relative_to(REPO))
for p in sorted(OUT.iterdir()):
    print(" ", p.name, f"({len(list(p.iterdir()))} files)" if p.is_dir() else f"({p.stat().st_size:,} B)")
print(f"图片 {len(local)}/{len(urls)} 张已本地化")

In [ ]:
from IPython.display import Markdown, display

display(Markdown(md_path.read_text(encoding="utf-8")[:4000]))   # 渲染开头看效果